## 准备数据

In [15]:
"""
在经典的MNIST手写数字数据集上，手动实现一个简单CNN的前向传播。
"""
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D

#只显示ERROR级别的日志，屏蔽INFO和WARNING信息，使控制台输出更简洁。
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    x = x.reshape(x.shape[0], 28, 28,1)
    x_test = x_test.reshape(x_test.shape[0], 28, 28,1)
    
    ds = tf.data.Dataset.from_tensor_slices((x, y))#构建Dataset对象，每个元素是一个(image,label)对
    ds = ds.map(prepare_mnist_features_and_labels)
    ds = ds.take(20000).shuffle(20000).batch(32)#只取了前20000个样本
    
    test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
    test_ds = test_ds.map(prepare_mnist_features_and_labels)
    test_ds = test_ds.take(20000).shuffle(20000).batch(20000)
    return ds, test_ds

def prepare_mnist_features_and_labels(x, y):
    x = tf.cast(x, tf.float32) / 255.0  #像素值归一化到[0,1]
    y = tf.cast(y, tf.int64) #标签转成int64
    return x, y

## 建立模型

In [ ]:
class myConvModel(keras.Model):
    def __init__(self):
        super(myConvModel, self).__init__()
        #定义卷积层：32 个卷积核+卷积核大小为5×5+激活函数为ReLU+padding='same' 保证输入输出尺寸相同（步长默认为1）
        self.l1_conv = Conv2D(32, (5, 5), activation='relu', padding='same')
        self.l2_conv = Conv2D(64, (5, 5), activation='relu', padding='same')

        self.pool = MaxPooling2D(pool_size=(2, 2), strides=2)

        self.flat = Flatten()#展平层，将多维特征图展平为一维向量，以便输入全连接层。

        self.dense1 = layers.Dense(100, activation='tanh') #第一个全连接层，输出100个神经元，激活函数为双曲正切（tanh）
        self.dense2 = layers.Dense(10) #第二个全连接层，输出10个神经元（对应MNIST的10个类别），通常后面会接softmax激活，这里没有显式指定，所以是线性输出（logits）。
    @tf.function
    def call(self, x):
        h1 = self.l1_conv(x)
        h1_pool = self.pool(h1)

        h2 = self.l2_conv(h1_pool)
        h2_pool = self.pool(h2)
        
        flat_h = self.flat(h2_pool)#展平为一维向量
        
        dense1 = self.dense1(flat_h)
        logits = self.dense2(dense1)
        return logits

model = myConvModel()
#必须有这一步，先跑一次前向把权重变量创建好，避免后面梯度追踪出现问题。
_ = model(tf.zeros((1, 28, 28, 1), dtype=tf.float32))
optimizer = optimizers.Adam()

## 定义loss以及train loop

In [17]:
@tf.function
def compute_loss(logits, labels):#平均损失
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):#准确率
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):#单步训练过程

    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = compute_loss(logits, y)

    # compute gradient
    grads = tape.gradient(loss, model.trainable_variables)
    # 过滤 None 梯度，避免 Keras 3 在空梯度时直接报错
    grads_and_vars = [(g, v) for g, v in zip(grads, model.trainable_variables) if g is not None]
    if not grads_and_vars:
        raise ValueError("All gradients are None. Please check model forward and loss graph.")
    # update to weights
    optimizer.apply_gradients(grads_and_vars)

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test_step(model, x, y):
    logits = model(x, training=False)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y) in enumerate(ds):
        loss, accuracy = train_one_step(model, optimizer, x, y)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())

    return loss, accuracy
def test(model, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y) in enumerate(ds):
        loss, accuracy = test_step(model, x, y)

        
    print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

    return loss, accuracy

# 训练

In [18]:
train_ds, test_ds = mnist_dataset()
for epoch in range(2):
    loss, accuracy = train(epoch, model, optimizer, train_ds)
loss, accuracy = test(model, test_ds)

epoch 0 : loss 2.3211927 ; accuracy 0.125
epoch 0 : loss 0.014275316 ; accuracy 1.0
epoch 1 : loss 0.09892893 ; accuracy 0.96875
epoch 1 : loss 0.002771899 ; accuracy 1.0
test loss 0.059150588 ; accuracy 0.9814
